In [37]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [38]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


In [106]:
block_size = 8
batch_size = 4
max_iters = 10000
#eval_interval = 250
learning_rate = 6e-3
eval_iters = 250

In [107]:
with open('../data/wizard_of_oz.txt', 'r', encoding='utf-8')as f:
    text = f.read()

chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [108]:
strng_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_strng = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [strng_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_strng[i] for i in l])

# encoded_hello = torch.tensor(encode('hello'),dtype=torch.long)
# decoded_hello = decode(encoded_hello.tolist())

data = torch.tensor(encode(text), dtype=torch.long)

In [109]:
n = int(0.8*len(data))

train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split=='train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    #print(ix)
    x= torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

x,y = get_batch('train')
print('inputs:')
print(x)
print('targets:')
print(y)

inputs:
tensor([[64, 58,  1, 61, 58, 71,  1, 69],
        [ 3, 72, 61, 58,  1, 76, 62, 65],
        [ 1, 54, 72,  1, 72, 68, 68, 67],
        [ 0,  0, 32, 58,  1, 60, 68, 73]])
targets:
tensor([[58,  1, 61, 58, 71,  1, 69, 54],
        [72, 61, 58,  1, 76, 62, 65, 65],
        [54, 72,  1, 72, 68, 68, 67,  1],
        [ 0, 32, 58,  1, 60, 68, 73,  1]])


In [110]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('When input is:',context,'target is:',target)

When input is: tensor([28]) target is: tensor(39)
When input is: tensor([28, 39]) target is: tensor(42)
When input is: tensor([28, 39, 42]) target is: tensor(39)
When input is: tensor([28, 39, 42, 39]) target is: tensor(44)
When input is: tensor([28, 39, 42, 39, 44]) target is: tensor(32)
When input is: tensor([28, 39, 42, 39, 44, 32]) target is: tensor(49)
When input is: tensor([28, 39, 42, 39, 44, 32, 49]) target is: tensor(1)
When input is: tensor([28, 39, 42, 39, 44, 32, 49,  1]) target is: tensor(25)


In [111]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train','val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X,Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [112]:
class BigramLanguagemodel(nn.Module):
    def __init__(self, vocab_size,hidden_size=128, dropout=0.2):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, hidden_size)# vocab_size)
       
        #hidden layers
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc_out = nn.Linear(hidden_size, vocab_size)
    

    def forward(self, index, targets=None):
        x = self.token_embedding_table(index) #is 3 dim ->(B,T,C)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.dropout(x)
        logits = self.fc_out(x)

        # ---- Compute loss only if targets are provided (training mode) ----
        if targets is None:
            loss = None
        else:
            # Flatten both tensors to feed into cross_entropy
            B,T,C = logits.shape  #T "time" is the sequence size or block_size, C "channel" is vocab_size
            logits = logits.view(B*T, C) # B*T act as total number of samples, C class scores
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets) #cross_Entropy expects input:(N,C), targets:(N,)
            
        return logits,loss

    def generate(self, index, max_new_tokens):
        #index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            #get predictions
            logits, loss = self.forward(index)
            #for generation (not training) targets is None so skips logits flattening -> logtis become (B,T,C)
            logits = logits[:,-1,:] #becomes (B, C), -1 gets only the last token from the time step
            #apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B,C), dim=-1 acts across C class channels
            #sample for distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            #append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1), concatenates across time sequence, if dim=0 it would stack batches
        return index


model = BigramLanguagemodel(vocab_size)
m = model.to(device) 

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


5*tNCzS;mS(!WiSrsL8oFBTR.gr_PHiR?Aj-oWWfVNT4wPWh6U",]h?Y!mmwja6WHn2mQL't;UFQ86vJeinSB;t19r0N8,rxzqfvMp'2v2vVJQ_n"2C
kGFD-.uX(]FC,,-J!R7I3zs6bsINAEOvw!?J7*Pu!W?hHe_O9rpJ_SFwt7VmrD"fY;bRXkAoXlNKW I'2[x'1A,PkYH8eH87o_YE"_eyhvT3nYOn[g0)F"6v)oO2g2PGyl!2N?;OnNfdWM6Dsc");lIO10Ob7,asR0N4
&_Tpl:gF[cyM68v?jdS1xvTb*(5Fk2y7V;7!a-_f]lEhMJ'dZJPzSXDxUTm.bH3iEy.pg]6e3nYc.1"9hMp!4x0M_-,JoW8TJKzULsLQxUjG;1GhC5EPQHZFZAdJF h)hO8Q)b_5QG]W0]BW?p 0ZkBG?nd!Zpg
)hjl
ZdRtwuA4!PfNzqndOMR"p.JsT*?;7?X17)yDboeP7&Km(rsVM?7ZId


In [113]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters ==0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.4f}, val loss: {losses['val']:.4f}")
        
    #sample a batch of data
    xb,yb = get_batch("train")

    #evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)#make grad star overnot accumulated
    loss.backward()
    optimizer.step()
    
print(loss.item())

step: 0, train loss: 4.4017, val loss: 4.4025
step: 250, train loss: 2.7272, val loss: 2.6941
step: 500, train loss: 2.6079, val loss: 2.6517
step: 750, train loss: 2.5832, val loss: 2.6243
step: 1000, train loss: 2.5518, val loss: 2.5938
step: 1250, train loss: 2.5959, val loss: 2.6107
step: 1500, train loss: 2.5437, val loss: 2.5800
step: 1750, train loss: 2.5336, val loss: 2.5914
step: 2000, train loss: 2.5332, val loss: 2.5708
step: 2250, train loss: 2.5565, val loss: 2.5735
step: 2500, train loss: 2.5389, val loss: 2.5648
step: 2750, train loss: 2.5337, val loss: 2.5880
step: 3000, train loss: 2.5377, val loss: 2.5803
step: 3250, train loss: 2.5728, val loss: 2.5757
step: 3500, train loss: 2.5325, val loss: 2.5710
step: 3750, train loss: 2.5422, val loss: 2.5843
step: 4000, train loss: 2.5316, val loss: 2.5728
step: 4250, train loss: 2.5061, val loss: 2.5570
step: 4500, train loss: 2.5430, val loss: 2.5625
step: 4750, train loss: 2.5092, val loss: 2.5507
step: 5000, train loss: 2.

In [114]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


BAR" t whe uper owhit trllo inithe hass stosth u Winen
ter goth tt Wuforany tospmm.

 s tor p'beanton
avetin busastangder be
jacabl."Pmpablom cry-athy, h can weas t, be sth! thngot iintton t  for camotrt boverden Ze mon

ooowotht tyscoud s."Wimerily itongro Ppetyos wlanrouudrre anevogsoonengrde cizmacrerend ," s atha e bo tha betato Whout ave tiporto S
weathoutlkmanire v lowocro inlon'n,"be 1Eustmanne g d pe boovoon,  afend achey ce an?
Hsorar
ut pe my mar h mousizabuutou Een wim fwee
 tsmupyofo
